In [1]:
!pip -q install imagehash pillow-heif

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 79.1 MB/s eta 0:00:00:00:0100:01


In [2]:
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
import hashlib, json, os, random, re, shutil, warnings

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm
import imagehash

try:
    from pillow_heif import register_heif_opener
    register_heif_opener()
except Exception as e:
    print("HEIF registration warning:", e)

ImageFile.LOAD_TRUNCATED_IMAGES = False
SEED = 42
random.seed(SEED); np.random.seed(SEED)
OUTPUT_DIR = Path('/kaggle/working/defakex_experiment_0_audit')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_EXTS = {'.jpg','.jpeg','.png','.webp','.bmp','.tif','.tiff','.heic','.heif','.dng'}
SPLIT_RATIOS = {'train':0.70, 'validation':0.10, 'locked_test':0.20}

candidate_roots = [
    Path('/kaggle/input/datasets/kashirhanif/defakex_dataset'),
    Path('kaggle/input/datasets/kashirhanif/defakex_dataset'),
    Path('/kaggle/input/defakex_dataset'),
    Path('/kaggle/input/defakex-dataset'),
]

def score_root(p):
    if not p.exists(): return -1
    return int(any(p.rglob('DefakeX_annotations_final.csv'))) * 100000 + sum(1 for x in p.rglob('*') if x.suffix.lower() in IMAGE_EXTS)

ranked = sorted(((score_root(p), p) for p in candidate_roots), reverse=True)
if not ranked or ranked[0][0] < 0:
    extras = [p.parent for p in Path('/kaggle/input').rglob('DefakeX_annotations_final.csv')]
    ranked = sorted(((score_root(p), p) for p in extras), reverse=True)
assert ranked and ranked[0][0] >= 0, 'Dataset not found. Attach it to this Kaggle notebook and verify the path.'
DATA_ROOT = ranked[0][1].resolve()
print('DATA_ROOT:', DATA_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)

DATA_ROOT: /kaggle/input/datasets/kashirhanif/defakex-dataset/DefakeX
OUTPUT_DIR: /kaggle/working/defakex_experiment_0_audit


In [3]:
image_paths = sorted(p for p in DATA_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
csv_candidates = list(DATA_ROOT.rglob('DefakeX_annotations_final.csv'))
if not csv_candidates:
    csv_candidates = [p for p in DATA_ROOT.rglob('*.csv') if 'annot' in p.name.lower()]
assert csv_candidates, 'No annotation CSV found inside the dataset.'
ANNOTATION_CSV = sorted(csv_candidates, key=lambda p: (p.name != 'DefakeX_annotations_final.csv', len(p.parts)))[0]

ann = pd.read_csv(ANNOTATION_CSV, dtype=str, keep_default_na=False)
ann.columns = [c.strip().lower() for c in ann.columns]
required = {'image_id','filepath','label'}
assert required.issubset(ann.columns), f'Missing required columns: {required-set(ann.columns)}'
ann['_row'] = np.arange(len(ann))
for c in ['label','source','generator','annotation_status','exclusion_reason','sha256','file_format']:
    if c not in ann: ann[c] = ''
    ann[c] = ann[c].astype(str).str.strip()
ann['label_norm'] = ann['label'].str.lower().replace({'ai':'fake','generated':'fake','1':'fake','0':'real'})
ann['excluded'] = ann['exclusion_reason'].ne('') | ann['annotation_status'].str.lower().isin({'exclude','excluded','invalid','failed'})
ann['needs_review'] = ann['annotation_status'].str.lower().isin({'pending','review','needs_review','unreviewed'})

print(f'Images found: {len(image_paths):,}')
print(f'Annotation rows: {len(ann):,}')
print('Annotation CSV:', ANNOTATION_CSV)
display(ann['label_norm'].value_counts(dropna=False).rename('rows').to_frame())

Images found: 6,003
Annotation rows: 6,003
Annotation CSV: /kaggle/input/datasets/kashirhanif/defakex-dataset/DefakeX/DefakeX_annotations_final.csv


,rows
label_norm,
fake,3002
real,3001


In [4]:
by_name = defaultdict(list)
by_rel = {}
for p in image_paths:
    by_name[p.name.lower()].append(p)
    by_rel[p.relative_to(DATA_ROOT).as_posix().lower()] = p

def clean_rel(s):
    s = str(s).replace('\\','/').strip()
    low = s.lower()
    markers = ['/real/','/gemini/','/chatgpt defakex/']
    for m in markers:
        if m in low:
            return s[low.index(m)+1:]
    return s.lstrip('/')

def resolve_row(row):
    candidates = [row.get('filepath',''), row.get('image_id','')]
    for raw in candidates:
        rel = clean_rel(raw)
        if rel.lower() in by_rel: return str(by_rel[rel.lower()]), 'relative_exact'
    names = []
    for raw in candidates:
        name = Path(str(raw).replace('\\','/')).name.lower()
        if name: names.append(name)
    matches = {str(p) for n in names for p in by_name.get(n,[])}
    if len(matches) == 1: return matches.pop(), 'unique_basename'
    if len(matches) > 1: return '', 'ambiguous_basename'
    return '', 'unmatched'

resolved = ann.apply(resolve_row, axis=1, result_type='expand')
ann[['resolved_path','path_resolution']] = resolved
display(ann['path_resolution'].value_counts().rename('rows').to_frame())

,rows
path_resolution,
relative_exact,6003


In [5]:
def inspect_image(path_str):
    p = Path(path_str)
    out = {'resolved_path':path_str, 'actual_width':'', 'actual_height':'', 'actual_format':'',
           'mode':'', 'file_bytes':'', 'actual_sha256':'', 'phash':'', 'read_error':''}
    try:
        h = hashlib.sha256()
        with p.open('rb') as f:
            for block in iter(lambda: f.read(1024*1024), b''): h.update(block)
        out['actual_sha256'] = h.hexdigest(); out['file_bytes'] = p.stat().st_size
        with Image.open(p) as im:
            im.load(); out['actual_width'], out['actual_height'] = im.size
            out['actual_format'] = (im.format or p.suffix.lstrip('.')).lower(); out['mode'] = im.mode
            out['phash'] = str(imagehash.phash(im.convert('RGB')))
    except Exception as e:
        out['read_error'] = f'{type(e).__name__}: {e}'
    return out

unique_paths = sorted(set(ann.loc[ann['resolved_path'].ne(''),'resolved_path']))
workers = min(12, max(2, (os.cpu_count() or 2)))
with ThreadPoolExecutor(max_workers=workers) as ex:
    scans = list(tqdm(ex.map(inspect_image, unique_paths), total=len(unique_paths), desc='Auditing images'))
scan = pd.DataFrame(scans)
audit = ann.merge(scan, on='resolved_path', how='left')
audit['included'] = ~audit['excluded']
print('Unreadable resolved images:', int(audit['read_error'].fillna('').ne('').sum()))

Auditing images:   0%|          | 0/6002 [00:00<?, ?it/s]

Unreadable resolved images: 1


In [6]:
def path_family(path):
    s = str(path).replace('\\','/').lower()
    if '/real/' in s: return 'real'
    if '/gemini/' in s: return 'gemini'
    if '/chatgpt defakex/' in s or '/chatgpt/' in s: return 'chatgpt'
    return 'other'

audit['path_family'] = audit['resolved_path'].map(path_family)
audit['path_label_expected'] = np.where(audit['path_family'].eq('real'),'real',np.where(audit['path_family'].isin(['gemini','chatgpt']),'fake',''))
audit['path_label_conflict'] = audit['path_label_expected'].ne('') & audit['label_norm'].ne(audit['path_label_expected'])
audit['sha_mismatch'] = audit['sha256'].ne('') & audit['actual_sha256'].ne('') & audit['sha256'].str.lower().ne(audit['actual_sha256'].str.lower())

resolved_set = set(audit['resolved_path'])
orphans = [str(p) for p in image_paths if str(p) not in resolved_set]
included = audit[audit['included']].copy()
sha_stats = included[included['actual_sha256'].ne('')].groupby('actual_sha256').agg(rows=('image_id','size'), labels=('label_norm','nunique')).reset_index()
ph_stats = included[included['phash'].ne('')].groupby('phash').agg(rows=('image_id','size'), labels=('label_norm','nunique')).reset_index()

issues=[]
def add(mask, kind, severity):
    for _,r in audit.loc[mask].iterrows():
        issues.append({'severity':severity,'issue':kind,'image_id':r['image_id'],'filepath':r['filepath'],'resolved_path':r['resolved_path']})
add(audit['included'] & audit['resolved_path'].eq(''), 'unresolved_included_annotation', 'critical')
add(audit['included'] & audit['read_error'].fillna('').ne(''), 'unreadable_included_image', 'critical')
add(audit['included'] & ~audit['label_norm'].isin(['real','fake']), 'invalid_label', 'critical')
add(audit['included'] & audit['path_label_conflict'], 'path_label_conflict', 'critical')
add(audit['included'] & audit['sha_mismatch'], 'annotation_sha256_mismatch', 'warning')
add(audit['needs_review'], 'annotation_needs_review', 'warning')
for p in orphans: issues.append({'severity':'warning','issue':'unannotated_image_file','image_id':'','filepath':'','resolved_path':p})
for h in sha_stats.loc[sha_stats['labels'].gt(1),'actual_sha256']:
    for _,r in included[included['actual_sha256'].eq(h)].iterrows(): issues.append({'severity':'critical','issue':'exact_duplicate_cross_label','image_id':r['image_id'],'filepath':r['filepath'],'resolved_path':r['resolved_path']})

issues_df = pd.DataFrame(issues, columns=['severity','issue','image_id','filepath','resolved_path'])
display(issues_df.groupby(['severity','issue']).size().rename('count').to_frame() if len(issues_df) else pd.DataFrame({'count':[]}))

,,count
severity,issue,
critical,unreadable_included_image,1
warning,unannotated_image_file,1


In [7]:
def identity_from_path(path, label, sha, image_id):
    if sha: duplicate_group = 'sha:'+sha
    else: duplicate_group = 'img:'+str(image_id)
    if label != 'real': return duplicate_group
    try:
        rel = Path(path).relative_to(DATA_ROOT).parts
        low = [x.lower() for x in rel]
        i = low.index('real')
        return 'real_identity:' + (rel[i+1] if i+1 < len(rel)-1 else Path(path).stem)
    except Exception: return duplicate_group

included['identity_group'] = [identity_from_path(p,l,h,i) for p,l,h,i in zip(included['resolved_path'],included['label_norm'],included['actual_sha256'],included['image_id'])]
included['stratum'] = np.where(included['label_norm'].eq('real'),'real',np.where(included['path_family'].eq('gemini'),'fake_gemini',np.where(included['path_family'].eq('chatgpt'),'fake_chatgpt','fake_other')))

existing_split_col = next((c for c in ['split','dataset_split','set'] if c in included.columns and included[c].astype(str).str.strip().ne('').any()), None)
if existing_split_col:
    included['final_split'] = included[existing_split_col].str.lower().replace({'val':'validation','valid':'validation','test':'locked_test'})
    split_origin = f'existing column: {existing_split_col}'
else:
    manifest_files=[]
    for p in DATA_ROOT.rglob('*.csv'):
        n=p.stem.lower()
        split = 'train' if re.search(r'(^|_)train($|_)',n) else ('validation' if re.search(r'(^|_)(val|validation)($|_)',n) else ('locked_test' if re.search(r'(^|_)(test|locked_test)($|_)',n) else None))
        if split and p != ANNOTATION_CSV: manifest_files.append((p,split))
    id_to_split={}; name_to_splits=defaultdict(set)
    for p,split in manifest_files:
        m=pd.read_csv(p,dtype=str,keep_default_na=False); m.columns=[c.strip().lower() for c in m.columns]
        if 'image_id' in m:
            for v in m['image_id'].astype(str): id_to_split[v.strip()]=split
        for col in ['resolved_path','filepath','path']:
            if col in m:
                for v in m[col].astype(str): name_to_splits[Path(v.replace('\\','/')).name.lower()].add(split)
    def lookup_existing(r):
        if str(r['image_id']).strip() in id_to_split: return id_to_split[str(r['image_id']).strip()]
        vals=name_to_splits.get(Path(str(r['resolved_path'])).name.lower(),set())
        return next(iter(vals)) if len(vals)==1 else ''
    from_manifests = included.apply(lookup_existing,axis=1) if manifest_files else pd.Series('',index=included.index)
    if manifest_files and from_manifests.ne('').all():
        included['final_split']=from_manifests
        split_origin='existing manifest files: '+', '.join(sorted(p.name for p,_ in manifest_files))
    else:
        if manifest_files: print('WARNING: existing manifests did not cover every included row; creating a proposal instead.')
        rng = np.random.default_rng(SEED)
        assignments={}
        for stratum, part in included.groupby('stratum'):
            sizes = part.groupby('identity_group').size().to_dict()
            groups = list(sizes); rng.shuffle(groups); groups.sort(key=lambda g:sizes[g], reverse=True)
            target={s:len(part)*r for s,r in SPLIT_RATIOS.items()}; used={s:0 for s in target}
            for g in groups:
                choice=max(target, key=lambda s:(target[s]-used[s])/max(target[s],1))
                assignments[(stratum,g)]=choice; used[choice]+=sizes[g]
        included['final_split']=[assignments[(s,g)] for s,g in zip(included['stratum'],included['identity_group'])]
        split_origin='deterministic group-aware proposal (seed 42)'

audit = audit.merge(included[['_row','identity_group','stratum','final_split']], on='_row', how='left')
print('Split source:', split_origin)
display(pd.crosstab([included['label_norm'],included['stratum']], included['final_split'], margins=True))

Split source: existing manifest files: DefakeX_test.csv, DefakeX_train.csv, DefakeX_validation.csv


final_split              locked_test  train  validation   All
label_norm stratum                                           
fake       fake_chatgpt          225   1051         225  1501
           fake_gemini           225   1051         225  1501
real       real                  448   2094         449  2991
All                              898   4196         899  5993

In [8]:
valid_split = included[included['final_split'].isin(SPLIT_RATIOS)]
sha_leak = valid_split[valid_split['actual_sha256'].ne('')].groupby('actual_sha256')['final_split'].nunique()
phash_leak = valid_split[valid_split['phash'].ne('')].groupby('phash')['final_split'].nunique()
identity_leak = valid_split[valid_split['label_norm'].eq('real')].groupby('identity_group')['final_split'].nunique()

gates = {
 'included_rows_resolve': int((included['resolved_path'].eq('')).sum()) == 0,
 'included_images_readable': int(included['read_error'].fillna('').ne('').sum()) == 0,
 'labels_are_real_or_fake': included['label_norm'].isin(['real','fake']).all(),
 'no_path_label_conflicts': int(included['path_label_conflict'].sum()) == 0,
 'no_exact_cross_label_duplicates': int((sha_stats['labels']>1).sum()) == 0,
 'no_exact_duplicate_split_leakage': int((sha_leak>1).sum()) == 0,
 'no_real_identity_split_leakage': int((identity_leak>1).sum()) == 0,
 'all_three_splits_present': set(SPLIT_RATIOS).issubset(set(valid_split['final_split'])),
 'no_pending_annotations': int(included['needs_review'].sum()) == 0,
}
gate_df = pd.DataFrame({'gate':gates.keys(),'pass':gates.values()})
display(gate_df.style.applymap(lambda x:'background-color:#c6efce' if x is True else ('background-color:#ffc7ce' if x is False else ''), subset=['pass']))
print('Perceptual-hash groups crossing splits (review candidates):', int((phash_leak>1).sum()))

/tmp/ipykernel_47/461190241.py:19: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  display(gate_df.style.applymap(lambda x:'background-color:#c6efce' if x is True else ('background-color:#ffc7ce' if x is False else ''), subset=['pass']))


,gate,pass
0,included_rows_resolve,True
1,included_images_readable,False
2,labels_are_real_or_fake,True
3,no_path_label_conflicts,True
4,no_exact_cross_label_duplicates,True
5,no_exact_duplicate_split_leakage,True
6,no_real_identity_split_leakage,True
7,all_three_splits_present,True
8,no_pending_annotations,True


Perceptual-hash groups crossing splits (review candidates): 1


In [9]:
for col in ['label_norm','source','generator','path_family','file_format','actual_format','framing','lighting','face_count','annotation_status']:
    if col in included.columns:
        print('\n###', col)
        display(included[col].replace('', '(blank)').value_counts(dropna=False).rename('count').to_frame().head(30))

print('Resolution summary:')
display(included[['actual_width','actual_height','file_bytes']].apply(pd.to_numeric, errors='coerce').describe().round(1))


### label_norm


,count
label_norm,
fake,3002
real,2991



### source


,count
source,
phone,1628
gemini,1501
chatgpt,1501
whatsapp,1363



### generator


...

,count
face_count,
1,3126
2+,2536
0,331



### annotation_status


,count
annotation_status,
automated_visual_review,5927
visually_reviewed,66


Resolution summary:


,actual_width,actual_height,file_bytes
count,5992.0,5992.0,5993.0
mean,2035.0,1765.1,2420790.7
std,1422.3,1242.9,2676419.2
min,256.0,240.0,11955.0
25%,1200.0,768.0,711531.0
50%,1408.0,1171.0,2100651.0
75%,2816.0,2430.8,2725662.0
max,10608.0,8160.0,27944102.0


In [10]:
manifest_cols = [c for c in ['image_id','filepath','resolved_path','label_norm','source','generator','face_count','framing','lighting','annotation_status','exclusion_reason','actual_width','actual_height','actual_format','file_bytes','actual_sha256','phash','identity_group','stratum','final_split','path_resolution','read_error'] if c in audit.columns]
audit[manifest_cols].to_csv(OUTPUT_DIR/'DefakeX_Experiment0_Audited_Manifest.csv', index=False)
for split, name in [('train','Train'),('validation','Validation'),('locked_test','Locked_Test')]:
    audit[audit['final_split'].eq(split)][manifest_cols].to_csv(OUTPUT_DIR/f'DefakeX_Experiment0_{name}.csv', index=False)
issues_df.to_csv(OUTPUT_DIR/'DefakeX_Experiment0_Issues.csv', index=False)

dups = included[included['actual_sha256'].isin(sha_stats.loc[sha_stats['rows'].gt(1),'actual_sha256'])]
dups[manifest_cols].to_csv(OUTPUT_DIR/'DefakeX_Experiment0_Exact_Duplicate_Groups.csv', index=False)

summary = {
 'experiment':'DefakeX Experiment 0', 'seed':SEED, 'data_root':str(DATA_ROOT), 'annotation_csv':str(ANNOTATION_CSV),
 'image_files_found':len(image_paths), 'annotation_rows':len(ann), 'included_rows':len(included), 'excluded_rows':int(audit['excluded'].sum()),
 'unannotated_image_files':len(orphans), 'split_origin':split_origin,
 'split_counts':valid_split.groupby(['final_split','label_norm']).size().unstack(fill_value=0).to_dict(orient='index'),
 'critical_issue_rows':int((issues_df['severity']=='critical').sum()), 'warning_issue_rows':int((issues_df['severity']=='warning').sum()),
 'perceptual_hash_groups_crossing_splits':int((phash_leak>1).sum()), 'gates':{k:bool(v) for k,v in gates.items()},
 'all_critical_gates_pass':bool(all(gates.values()))
}
(OUTPUT_DIR/'DefakeX_Experiment0_Summary.json').write_text(json.dumps(summary, indent=2))
(OUTPUT_DIR/'README.txt').write_text('DefakeX Experiment 0 audit outputs. The Locked_Test manifest must remain untouched by training, threshold selection, calibration, and model selection.\n')

zip_path = Path('/kaggle/working/DefakeX_Experiment0_Audit_Outputs')
shutil.make_archive(str(zip_path), 'zip', OUTPUT_DIR)
print(json.dumps(summary, indent=2))
print('\nZIP:', str(zip_path)+'.zip')
from IPython.display import FileLink, display
display(FileLink(str(zip_path)+'.zip'))

{
  "experiment": "DefakeX Experiment 0",
  "seed": 42,
  "data_root": "/kaggle/input/datasets/kashirhanif/defakex-dataset/DefakeX",
  "annotation_csv": "/kaggle/input/datasets/kashirhanif/defakex-dataset/DefakeX/DefakeX_annotations_final.csv",
  "image_files_found": 6003,
  "annotation_rows": 6003,
  "included_rows": 5993,
  "excluded_rows": 10,
  "unannotated_image_files": 1,
  "split_origin": "existing manifest files: DefakeX_test.csv, DefakeX_train.csv, DefakeX_validation.csv",
  "split_counts": {
    "locked_test": {
      "fake": 450,
      "real": 448
    },
    "train": {
      "fake": 2102,
      "real": 2094
    },
    "validation": {
      "fake": 450,
      "real": 449
    }
  },
  "critical_issue_rows": 1,
  "warning_issue_rows": 1,
  "perceptual_hash_groups_crossing_splits": 1,
  "gates": {
    "included_rows_resolve": true,
    "included_images_readable": false,
    "labels_are_real_or_fake": true,
    "no_path_label_conflicts": true,
    "no_exact_cross_label_duplicates

/kaggle/working/DefakeX_Experiment0_Audit_Outputs.zip